# catnip — stage 1 training on Kaggle

Trains YOLO26n on the SAHI-sliced manga face/body dataset.  Reads the
sliced dataset from `/kaggle/input/catnip-stage1-sliced/`, writes model
weights and run curves to `/kaggle/working/catnip-data-local/`, and
publishes them to a private Kaggle dataset at session end so they
survive the 12 h session cap.

**Session config (right sidebar):**

- Accelerator: GPU T4 x2
- Internet: ON
- Persistence: Files only
- Attach dataset: `catnip-stage1-sliced`

Re-run cell 2 to resume from `last.pt` after the 12 h cap.

In [ ]:
%pip install --quiet \
    "ultralytics>=8.4,<9" \
    "pydantic>=2.11,<3"        "pydantic-settings>=2.11,<3" \
    "omegaconf>=2.3,<3"        "sahi>=0.11.36,<0.13" \
    "opencv-python>=4.10,<5"   "python-dotenv>=1,<2" \
    "scikit-learn>=1.7,<2"     "manga109api>=0.3.1,<0.4" \
    "kagglehub>=0.3,<1"

# Kaggle secret → env var for kagglehub / kaggle_publish.py.
from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
import os
os.environ["KAGGLE_API_TOKEN"] = _secrets.get_secret("KAGGLE_API_TOKEN")

# Clone the repo.  -b refactor/reID matches the Colab notebook branch
# so train behaviour is identical.  /kaggle/working/ is the only
# writable area on Kaggle — the repo lives there.
REPO_DIR = "/kaggle/working/catnip"
BRANCH = "refactor/reID"
!git clone -b {BRANCH} https://github.com/rifusaki/catnip.git {REPO_DIR}
%cd {REPO_DIR}
import sys; sys.path.insert(0, REPO_DIR)

# --- Materialise the dataset into a writable working dir ---
# /kaggle/input/ is read-only.  The trainer needs to write:
#   - a patched dataset.yaml (absolute `path:`)
#   - per-split *.cache files (fast-image-index; ~2.8 GB total)
#   - model/best.pt, runs/detect/stage1/*, etc.
# All of these go under paths.data (= $CATNIP_DATA), so we point that
# at /kaggle/working/catnip-data-local and copy the dataset in.
#   - 14 GB copy @ 1-2 GBps = 7-15 s
#   - 14 GB dataset + 2.8 GB caches + ~50 MB outputs = ~17 GB
#   - 20 GB working-disk cap leaves a 3 GB buffer.
from pathlib import Path
import shutil
SRC = Path("/kaggle/input/catnip-stage1-sliced")
DST = Path("/kaggle/working/catnip-data-local")
if not DST.exists():
    print(f"Copying {SRC} → {DST} (≈14 GB, ~10-15 s at 1-2 GBps)...")
    shutil.copytree(SRC, DST, dirs_exist_ok=False)
    print("Done.")
else:
    print(f"{DST} already present — skipping copy.")
os.environ["CATNIP_DATA"] = str(DST)

# Sanity: confirm GPU + dataset present
!nvidia-smi -L || true
!ls {DST}/training/stage1_sliced/dataset.yaml
print(f"CATNIP_DATA={os.environ['CATNIP_DATA']}")

In [ ]:
# --- Train (or resume) Stage 1 ---
#
# We run scripts/train/stage1.main() in-process for the same reason as
# the Colab notebook: subprocess stdout/stderr gets lost in cell output.
#
# Resume behaviour: if /kaggle/working/catnip-data-local/runs/detect/
# stage1/weights/last.pt exists (from a prior session, restored from
# the output dataset via the side-panel "Add data"), we pass it via
# --resume so the model picks up at the next epoch.  Otherwise this is
# a fresh run.
import os, sys, yaml
from pathlib import Path

REPO = "/kaggle/working/catnip"
sys.path.insert(0, REPO); os.chdir(REPO)
os.environ.setdefault("CATNIP_DATA", "/kaggle/working/catnip-data-local")

# Patch dataset.yaml — the sliced YAML uses `path: .` but ultralytics
# 8.4.x resolves it relative to CWD, not the YAML dir.  Set the
# absolute path so the trainer finds the images regardless of CWD.
_yaml_path = Path(os.environ["CATNIP_DATA"]) / "training" / "stage1_sliced" / "dataset.yaml"
_d = yaml.safe_load(_yaml_path.read_text())
_d["path"] = str(_yaml_path.parent.resolve())
_yaml_path.write_text(yaml.dump(_d, default_flow_style=False))
print(f"dataset.yaml path → {_d['path']}")

# Detect a resume target.  Kaggle's /kaggle/working/ is wiped at
# session end, so to resume a prior session: attach the most recent
# `catnip-stage1-output` version as a *second* dataset, copy its
# weights/ into the canonical location below, and re-run this cell.
RESUME = Path("/kaggle/working/catnip-data-local/runs/detect/stage1/weights/last.pt")
if not RESUME.exists():
    RESUME = None
    print("No prior checkpoint at last.pt — starting fresh.")
else:
    print(f"Resuming from {RESUME}")

sys.argv = [
    "stage1.py",
    "--override", "training.stage1.device=cuda",
    "--override", "training.stage1.workers=4",
    "--override", "training.stage1.batch=64",
    "--override", "training.stage1.epochs=24",  # ≈12 h on T4; raise for longer chunks
    "--verbose",
]
if RESUME is not None:
    sys.argv += ["--resume", str(RESUME)]

from scripts.train import stage1
try:
    stage1.main()
except SystemExit as e:
    print(f"\nTraining exited with code {e.code}.")
    raise

In [ ]:
# --- Publish outputs to a private Kaggle dataset ---
#
# /kaggle/working/ is wiped at session end, so the only way to keep
# best.pt + results.csv + plots is to push them to a private dataset
# version.  scripts/kaggle_publish.py handles staging, cache cleanup
# (the .cache files are ~2.8 GB of derived state, not deliverables),
# and now uses kagglehub.dataset_upload() with KAGGLE_API_TOKEN.
!python /kaggle/working/catnip/scripts/kaggle_publish.py \
    --output-dir /kaggle/working/catnip-data-local \
    --dataset-slug catnip-stage1-output \
    --version-notes "session $(date -u +%Y-%m-%dT%H:%MZ), batch=64, 24 epochs"